In [0]:
%ls ../data/flight-data/csv/


# Building a Spark Session

Every notebook needs a `SparkSession` — the single entry point for the DataFrame and SQL APIs. `getOrCreate()` reuses the existing session if one is already running.


Code:
```python
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("learning-spark").getOrCreate()
```

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("learning-spark").getOrCreate()

# Setup — Paths & Imports

Shared constants used across this notebook:

* `setup_data_dirs(dir_name)` — reusable helper: copies `data/{dir_name}` from the repo into a UC volume and creates its matching output volume (`{dir_name}` → `/Volumes/workspace/default/{dir_name}`, outputs → `/Volumes/workspace/default/{dir_name}-output`)
* `FLIGHT_DATA_DIR` / `OUTPUT_DIR` — dataset and write-output paths, derived from the helper
* `pprint` — pretty-printer for readable `dtypes` and schema output

In [0]:
import shutil
from pathlib import Path

from pprint import pprint

# Reusable setup — Unit Catalog (UC) volumes are the only workspace locations Spark can both
# READ from and WRITE to. This helper copies a dataset directory from the git
# repo into a UC volume and prepares its matching output directory.
REPO_DATA_ROOT = Path("/Workspace/Repos/mayurkedari16@gmail.com/pyspark-databricks/data")
VOLUME_ROOT = Path("/Volumes/workspace/default")


def setup_data_dirs(
    dir_name: str,
    data_volume: str | None = None,
    output_volume: str | None = None,
) -> tuple[str, str]:
    """
    Copy <repo>/data/{dir_name} into a UC volume and create its output directory.

    * data:   {REPO_DATA_ROOT}/{dir_name}  ->  {VOLUME_ROOT}/{data_volume or dir_name}
    * output:                                 {VOLUME_ROOT}/{output_volume or {dir_name}-output}

    Creates the volumes if they don't exist (CREATE VOLUME IF NOT EXISTS) and is
    idempotent — re-running safely overwrites the staged copies.

    Returns (data_dir, output_dir) as strings; data_dir ends in "/"
    for path concatenation (FLIGHT_DATA_DIR + "csv/*.csv").
    """
    src = Path(REPO_DATA_ROOT) / dir_name
    if not src.is_dir():
        raise FileNotFoundError(f"No dataset directory at {src}")

    data_volume = data_volume or dir_name
    output_volume = output_volume or f"{dir_name}-output"

    # A volume must exist in Unity Catalog before its /Volumes path is usable
    spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.default.`{data_volume}`")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.default.`{output_volume}`")

    data_dir = Path(VOLUME_ROOT) / data_volume
    output_dir = Path(VOLUME_ROOT) / output_volume

    shutil.copytree(src, data_dir, dirs_exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    return data_dir.as_posix() + "/", output_dir.as_posix() + "/"

In [0]:
# Wire up this notebook's dataset using the reusable helper — copies
# data/flight-data from the repo into the flight_data volume (idempotent)
# and points OUTPUT_DIR at the flight_output volume.
FLIGHT_DATA_DIR, OUTPUT_DIR = setup_data_dirs(
    "flight-data",
    data_volume="flight_data",
    output_volume="flight_output",
)

print("Dataset Dir:", FLIGHT_DATA_DIR)
print("Outputs Dir:", OUTPUT_DIR)

In [0]:
%sh ls /Volumes/workspace/default/

# Reading a CSV File

## Method 1 — Generic `DataFrameReader`

> [!TIP]
> **💡 Recommended approach.** Prefer the generic reader (not the Method 2 shorthand) for all file reads — it keeps your code consistent across formats. It supports the common built-in file formats — `csv`, `json`, `parquet`, `orc`, `avro`, `text` — and third-party source systems such as **Kafka**, **BigQuery**, **Snowflake**, and **Delta Lake**.

```python
flight_data_csv_format_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(FLIGHT_DATA_DIR + "csv/*.csv")
)

flight_data_csv_format_df.show(5)
```



In [0]:
flight_data_csv_format_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(FLIGHT_DATA_DIR + "csv/*.csv")
)

flight_data_csv_format_df.show(5)

# Reading a CSV File

## Method 2 — Format-Specific `DataFrameReader`

Shorthand for Method 1 — `spark.read.csv(...)` instead of `spark.read.format("csv").load(...)`. Options are passed as keyword arguments rather than chained `.option()` calls.

> [!NOTE]
> Every common format has an equivalent shorthand: `spark.read.json(...)`, `spark.read.parquet(...)`, `spark.read.orc(...)`, `spark.read.text(...)`.

```python
flight_csv_df = (
    spark.read
    .csv(
        FLIGHT_DATA_DIR + "csv/*.csv",
        header=True,
        inferSchema=True,
        mode="FAILFAST",
    )
)

flight_csv_df.show(5)
```



In [0]:
flight_csv_df = (
    spark.read
    .csv(
        FLIGHT_DATA_DIR + "csv/*.csv", 
        header=True, 
        inferSchema=True,
        mode="FAILFAST",
    )
)

flight_csv_df.show(5)

# Inspecting the DataFrame — Columns, Schema & Data Types

Three ways to inspect a DataFrame:

| Method | Returns |
| --- | --- |
| `df.dtypes` | List of `(column, type)` tuples — quick type check |
| `df.schema` | Full `StructType` object — nested structure, nullability & metadata |
| `df.printSchema()` | Human-readable tree of `.schema` |

In [0]:
# print only column/field name and its respective data type
print(flight_csv_df.dtypes)
print("-" * 50)

# prints schema as a StructType and its nested structure - Pyspark Object DataTypes
print(flight_csv_df.schema)

# Print Schema
flight_csv_df.printSchema()


In [0]:
flight_csv_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "FAILFAST")
    .csv(FLIGHT_DATA_DIR + "csv/*.csv")
)
flight_csv_df.show(5)

pprint(flight_csv_df.dtypes)
print("-" * 50)
pprint(flight_csv_df.schema)


In [0]:
flight_df_lowercase_cols = flight_data_csv_format_df.toDF(*[col.lower() for col in flight_data_csv_format_df.columns])

flight_df_lowercase_cols.show(5)



# Defining a Schema (CSV)

> [!IMPORTANT]
> **CSV files have no embedded schema** — without one, Spark reads **every column as `string`**. Two ways to enforce a schema (both replace `inferSchema`):

| Method | How | Notes |
| --- | --- | --- |
| 1 — `StructType` objects | `StructType([StructField("col", StringType(), True), ...])` | Explicit; supports per-column **metadata** |
| 2 — DDL string | `"dest_country STRING, count INT"` | Compact; no metadata support |

Pass either via `.schema(...)` and keep `.option("inferSchema", False)`.





## Method 1 — `StructType` Objects (PySpark data types)

Explicit PySpark objects imported from `pyspark.sql.types` — `StructType`, `StructField`, plus type objects like `StringType()` and `IntegerType()`.

**Syntax — quick reference**

```python
StructType([StructField(...), StructField(...), ...])

StructField(
    name,        # str — column name
    dataType,    # e.g. StringType(), IntegerType() — must be instantiated
    nullable,    # True/False — allows null values (default True)
    metadata={}, # optional dict — e.g. {"description": "..."}
)
```

**Minimal example**

```python
StructType([
    StructField("count", IntegerType(), True, metadata={"description": "Flight Count"}),
])
```

> [!NOTE]
> The third `StructField` argument is **nullability**. A fourth, optional `metadata` dict is supported **only** in this method — not in DDL strings.

```python
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
)

flight_schema_struct = StructType([
    StructField("dest_country", StringType(), True),
    StructField("origin_country", StringType(), True),
    StructField("count", IntegerType(), True),
])

flight_data_csv_format_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .schema(flight_schema_struct)
    .load(FLIGHT_DATA_DIR + "csv/*.csv")
)

flight_data_csv_format_df.show(3)
```

In [0]:
from pyspark.sql.types import (
    StructType, 
    StructField,
    IntegerType,
    StringType,
)


# Method 1 - Schema Enforcement using StructType (No inferSchema)
print("=" * 50)
print("Schema Enforcement using StructType")
print("=" * 50)

flight_schema_struct = StructType([
    StructField("dest_country", StringType(), True),
    StructField("origin_country", StringType(), True),
    StructField("count", IntegerType(), True),
])

# Generic DataFrameReader
flight_data_csv_format_df = (
    spark.read
    .format("csv")
    .option("header", False)
    .option("skipRows", 1) # Skips the first N lines of the file 
    .option("inferSchema", False)
    .schema(flight_schema_struct)
    .load(FLIGHT_DATA_DIR + "csv/*.csv")
)

flight_data_csv_format_df.show(3)



## Method 2 — DDL Schema String

A single compact string: `"column1 TYPE1, column2 TYPE2, ..."` — readable and import-free.

> [!NOTE]
> DDL strings **don't support per-column `metadata`** — use Method 1 (`StructType`) if you need metadata.

**Example:**

```python
schema_ddl = "dest_country STRING, origin_country STRING, count INT"

flight_data_csv_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .schema(schema_ddl)
    .csv(FLIGHT_DATA_DIR + "csv/*.csv")
)
flight_data_csv_df.show(3)
```

In [0]:
# Method 2 - Schema Enforcement using DDL (No inferSchema)

print("=" * 50)
print("Schema Enforcement using DDL")
print("=" * 50)
schema_ddl = "dest_country STRING, origin_country STRING, count INT"

# File Scecific DataFrameReader
flight_data_csv_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .schema(schema_ddl)
    .csv(FLIGHT_DATA_DIR + "csv/*.csv")

)
flight_data_csv_df.show(3)


# Reading a JSON File

> [!NOTE]
> JSON is **semi-structured** — field names come from the JSON keys, and Spark **infers data types automatically** on read (no `inferSchema` option needed). Options like `dateFormat` control how date/time strings are parsed. To control types exactly, enforce a schema — see the next section.

In [0]:
flight_json_df = (
    spark.read.format("json")
    .option("dateFormat", "yyyy-MM-dd")
    .load(FLIGHT_DATA_DIR + "json/*.json")
)

flight_json_df.show(10)

## Schema Enforcement for JSON Files — `StructType` & DDL

Same two schema-definition methods as the CSV section, now applied to JSON files:

* **`StructType`** — explicit Python objects; also the only way to attach column **metadata** (e.g. `description`)
* **DDL string** — compact `"col1 TYPE, col2 TYPE"` syntax; no metadata support

> [!IMPORTANT]
> Field names must **exactly match the keys in the JSON file** — matching is case-sensitive. A mismatched name does not raise an error: the field is simply read as `null`, and extra JSON keys are ignored.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructField,
    StructType,
    StringType,
    IntegerType,
)

spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

FLIGHT_DATA_DIR = "/Volumes/workspace/default/flight_data/"


print("=" * 50)
print("Schema Enforcement using StructType")
print("=" * 50)

schema = StructType([
    StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
    StructField("DEST_COUNTRY_NAME", StringType(), True),
    StructField("count", IntegerType(), True, metadata={
        "description": "Flight Count",
    }),
])

options = {
    "dateFormat": "yyyy-MM-dd",
    "multiLine": False,
    "encoding": "utf-8",
    "lineSep": "\n",
}
json_df = (
    spark.read
    .format("json")
    .options(**options)
    .schema(schema)
    .load(FLIGHT_DATA_DIR + "json/*.json")
)

json_df.show(5)

print("=" * 50)
print("Print Schema")
print("=" * 50)

json_df.printSchema()

print("=" * 50)
print("Schema Enforcement using DDL")
print("=" * 50)

schema_ddl = "ORIGIN_COUNTRY_NAME STRING, DEST_COUNTRY_NAME STRING, count INT"
json_ddl_df = (
    spark.read
    .format("json")
    .options(**options)
    .schema(schema_ddl)
    .load(FLIGHT_DATA_DIR + "json/*.json")
)

json_ddl_df.show(3)

In [0]:
from pprint import pprint

# 1) Access the StructField by name — .metadata is a plain dict
print(json_df.schema["count"].metadata)

# 2) Loop over every field: name, type, and metadata
for field in json_df.schema.fields:
    print(f"{field.name:22s} {field.dataType.simpleString():8s} metadata={field.metadata}")

# 3) Metadata is also part of the JSON representation of the schema
pprint(json_df.schema.jsonValue()["fields"][2])

# 4) Add/update metadata on an existing DataFrame (returns a new DataFrame)
json_df_with_meta = json_df.withMetadata("count", {"description": "Flight Count", "comment": "Total number of flights"})
json_df_with_meta.printSchema()

# Overcoming the JSON Key vs Column Name Mismatch

The JSON reader has **no key→column mapping option**, so field names in the enforced schema must exactly match the JSON keys (see the note above). To get your target column names, use a **two-step pattern**:

1. Enforce the schema using the **exact JSON keys** (as in the cells above)
2. Rename to the target names **immediately after the read**

| Option | Code | Behavior |
| --- | --- | --- |
| A — `withColumnRenamed` | `.withColumnRenamed("count", "flight_count")` | Renames one column at a time; **silent no-op** if the name doesn't exist |
| B — `toDF` | `.toDF("origin_country", "dest_country", "flight_count")` | Renames **all** columns positionally — breaks if the source column order changes |
| C — `select` + alias | `.select(F.col("count").alias("flight_count"), ...)` | Renames, reorders, and transforms in one expression |

> [!WARNING]
> Renaming creates **new fields with empty metadata** — any `StructField` metadata (e.g. `description`) is lost. Re-attach it with `withMetadata(...)` if needed.


In [0]:
# Overcoming the JSON key vs column name mismatch
# Step 1: schema uses the EXACT JSON keys (see cell above — json_ddl_df)
# Step 2: rename to target names right after the read

# Option A — withColumnRenamed: rename one column at a time (no-op if name not found)
renamed_df = (
    json_ddl_df
    .withColumnRenamed("ORIGIN_COUNTRY_NAME", "origin_country")
    .withColumnRenamed("DEST_COUNTRY_NAME", "dest_country")
    .withColumnRenamed("count", "flight_count")
)

# Option B — toDF: rename ALL columns at once, positionally (order matters!)
# renamed_df = json_ddl_df.toDF("origin_country", "dest_country", "flight_count")

# Option C — select with alias: rename + reorder/transform in one expression
# from pyspark.sql import functions as F
# renamed_df = json_ddl_df.select(
#     F.col("ORIGIN_COUNTRY_NAME").alias("origin_country"),
#     F.col("DEST_COUNTRY_NAME").alias("dest_country"),
#     F.col("count").alias("flight_count"),
# )

renamed_df.printSchema()
renamed_df.show(3, truncate=False)

# Reading Parquet Files

> [!NOTE]
> **Self-describing format — no schema needed.** Parquet embeds the schema in the file footer metadata, so Spark reads it automatically. **Do not** pass `.schema(...)` or the `inferSchema` option — they're unnecessary (and `inferSchema` is simply ignored). Only text formats like CSV and JSON rely on schema enforcement or inference.

You *can* optionally pass a schema to override the file's own — e.g. to prune columns or force a type — but in normal use, just read:

```python
flight_parquet_df = spark.read.format("parquet").load(FLIGHT_DATA_DIR + "parquet/*.parquet")
```

In [0]:
# No .schema(...) and no inferSchema — the schema comes from the file footer
flight_parquet_df = (
    spark.read
    .format("parquet")
    .load(FLIGHT_DATA_DIR + "parquet/*.parquet")
)

flight_parquet_df.printSchema()
flight_parquet_df.show(3, truncate=False)

# Reading ORC Files

> [!NOTE]
> Same as Parquet — **self-describing format, no schema needed.** ORC embeds the schema in the file footer metadata, so no `.schema(...)` and no `inferSchema` required.

```python
flight_orc_df = spark.read.format("orc").load(FLIGHT_DATA_DIR + "orc/*.orc")
```

In [0]:
# No .schema(...) and no inferSchema — the schema comes from the file footer
flight_orc_df = (
    spark.read
    .format("orc")
    .load(FLIGHT_DATA_DIR + "orc/*.orc")
)

flight_orc_df.printSchema()
flight_orc_df.show(3, truncate=False)


### Create Avro files from Parquet files

In [0]:
# Direct write — the dataset lives in a UC volume (flight_data),
# the only location type on this workspace that Spark can WRITE to
(
    flight_parquet_df.write
    .format("avro")
    .mode("overwrite")
    .save(FLIGHT_DATA_DIR + "avro/")
)

print("Avro written to:", FLIGHT_DATA_DIR + "avro/")


# Reading Avro Files

> [!NOTE]
> Avro is also **self-describing — no schema needed.** The schema is stored as JSON in each file's header, so Spark reads it automatically. Avro support is built into Databricks (`spark-avro` is bundled with the runtime).

```python
flight_avro_df = spark.read.format("avro").load(FLIGHT_DATA_DIR + "avro/*.avro")
```

In [0]:
# No .schema(...) and no inferSchema — Avro is self-describing.

flight_avro_df = (
    spark.read
    .format("avro")
    .load(FLIGHT_DATA_DIR + "avro/*.avro")
)

flight_avro_df.printSchema()
flight_avro_df.show(3, truncate=False)

---

> **⬆️ READ Operations** — above &nbsp;·&nbsp; **⬇️ WRITE Operations** — below

---



# Write Operations

The writer mirrors the reader: `df.write.format("<format>").mode(...).save(<path>)`, with the same format-specific shorthands (`df.write.csv(...)`, `df.write.parquet(...)`, ...).

> [!IMPORTANT]
> `mode` controls what happens when the target path already exists:

| Mode | Behavior when path exists |
| --- | --- |
| `error` (default) | Throw an exception |
| `append` | Add new files alongside existing data |
| `overwrite` | Replace existing data entirely |
| `ignore` | Silently do nothing |

> [!WARNING]
> Spark writes **many part files** (one per task), not a single file — use `coalesce(1)` before writing if you need a single file, and `partitionBy(...)` to lay data out in folder partitions.

> [!WARNING]
> **This workspace:** Spark can only **write** to UC volumes (`/Volumes/...`) — `/Workspace/Repos` is read-only for Spark writes and DBFS root is disabled. The dataset (`flight_data` volume) and the write output location (`flight_output` volume) therefore both live in UC volumes — the one location type that supports **both reads and writes**, so no staging or copying is needed.

In [0]:
# No staging needed anymore — FLIGHT_DATA_DIR (dataset) and OUTPUT_DIR (write outputs)
# are both UC volumes, defined once in the Setup cell at the top of the notebook.
# UC volumes are the only locations on this workspace that Spark can both READ from and WRITE to.


## Writing Data to CSV Files

> [!NOTE]
> Write options mirror the read options — `header`, `sep`, `quote`, `escape`, `nullValue`, `dateFormat`, `encoding`, ... All values are serialized as **plain text**, so types are not preserved (unlike Parquet/ORC/Avro).

In [0]:
# CSV write — options mirror the reader
DEST = OUTPUT_DIR + "/csv/"

(
    flight_data_csv_format_df
    .write
    .format("csv")
    .mode("overwrite")
    .option("header", True)
    .option("delimiter", ",")
    .save(DEST)
)

print("CSV written to:", DEST)

## Writing Data to JSON Files

> [!NOTE]
> JSON writes serialize **one JSON object per row** — column names become the JSON keys. No `header` option (it doesn't apply); `dateFormat`/`timestampFormat` control how date/time values are formatted.

```python
flight_json_df.write.format("json").mode("overwrite").save(OUTPUT_DIR + "/json/")
```

In [0]:
# JSON write — one JSON object per row; column names become the keys
DEST = OUTPUT_DIR + "/json/"

(
    flight_json_df
    .write
    .format("json")
    .mode("overwrite")
    .option("dateFormat", "yyyy-MM-dd")
    .save(DEST)
)

print("JSON written to:", DEST)

## Writing Data to Parquet Files

> [!NOTE]
> Self-describing on **write** too — the full Spark schema (including column **metadata**) is stored in the file footer and preserved on round-trip. Full type fidelity; the recommended default for storage.

```python
flight_parquet_df.write.format("parquet").mode("overwrite").save(OUTPUT_DIR + "/parquet/")
```

In [0]:
# Parquet write — schema (and column metadata) stored in the footer
DEST = OUTPUT_DIR + "/parquet/"

(
    flight_parquet_df
    .write
    .format("parquet")
    .mode("overwrite")
    .save(DEST)
)

print("Parquet written to:", DEST)

## Writing Data to ORC Files

> [!NOTE]
> Same as Parquet — the schema is written into the file **footer** and all types are preserved on round-trip.

```python
flight_orc_df.write.format("orc").mode("overwrite").save(OUTPUT_DIR + "/orc/")
```

In [0]:
# ORC write — schema stored in the footer, same as Parquet
DEST = OUTPUT_DIR + "/orc/"

(
    flight_orc_df
    .write
    .format("orc")
    .mode("overwrite")
    .save(DEST)
)

print("ORC written to:", DEST)

## Writing Data to Avro Files

> [!NOTE]
> The Spark schema is serialized **as JSON into each file's header**. Handy here: this dataset has no Avro source, so we write the Parquet DataFrame as Avro — which also gives the Avro *read* cell above its first data.

```python
flight_parquet_df.write.format("avro").mode("overwrite").save(OUTPUT_DIR + "/avro/")
```

In [0]:
# No Avro source in the original dataset — write the Parquet DataFrame as Avro,
# then read it back (gives the Avro read cell above its first data)
DEST = OUTPUT_DIR + "/avro/"

(
    flight_parquet_df
    .write
    .format("avro")
    .mode("overwrite")
    .save(DEST)
)

print("Avro written to:", DEST)

# Read the written Avro back — proves the round-trip (volumes are immediately consistent)
spark.read.format("avro").load(DEST).show(3)

# ==============================

## Additional Write Operation Practices

# ==============================

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, 
    StructField, 
    IntegerType, 
    StringType, 
    DoubleType,
)


spark = SparkSession.builder.appName("write-operations").getOrCreate()


In [0]:
data = [
        # ("id", "name", "age", "salary", "country", "gender"),
        (1, "Manish", 26, 75000, "INDIA", "m"),
        (2, "Nikita", 23, 100000, "USA", "f"),
        (3, "Pritam", 22, 150000, "INDIA", "m"),
        (4, "Prantosh", 17, 200000, "JAPAN", "m"),
        (5, "Vikash", 31, 300000, "USA", "m"),
        (6, "Rahul", 55, 300000, "INDIA", "m"),
        (7, "Raju", 67, 540000, "USA", "m"),
        (8, "Praveen", 28, 70000, "JAPAN", "m"),
        (9, "Dev", 32, 150000, "JAPAN", "m"),
        (10, "Sherin", 16, 25000, "RUSSIA", "f"),
        (11, "Ragu", 12, 35000, "INDIA", "f"),
        (12, "Sweta", 43, 200000, "INDIA", "f"),
        (13, "Raushan", 48, 650000, "USA", "m"),
        (14, "Mukesh", 36, 95000, "RUSSIA", "m"),
        (15, "Prakash", 52, 750000, "INDIA", "m"),
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", DoubleType(), True),
    StructField("country", StringType(), True),
    StructField("gender", StringType(), True),
])
df = spark.createDataFrame(data, schema=schema, )
df.show(5)

In [0]:
# Wire up this notebook's dataset using the reusable helper — copies
# data/flight-data from the repo into the flight_data volume (idempotent)
# and points OUTPUT_DIR at the flight_output volume.
WRITE_OPS_DATA_DIR, WRITE_OPS_OUTPUT_DIR = setup_data_dirs(
    "flight-data",
    data_volume="write_ops_data",
    output_volume="write_ops_data_output",
)

print("Dataset Dir:", WRITE_OPS_DATA_DIR)
print("Outputs Dir:", WRITE_OPS_OUTPUT_DIR)

In [0]:
(
    df.write.format("csv")
            .mode("overwrite")
            .option("path", WRITE_OPS_OUTPUT_DIR + "csv_out/")
            .option("header", True)
            .save()
 
 )

In [0]:
%sh

ls -ltr /Volumes/workspace/default/write_ops_data_output/csv_out/part-* | wc -l && echo '=== Data ===' && ls -1 /Volumes/workspace/default/write_ops_data_output/csv_out/part-* 
